In [ ]:
print("Hello, VS Code!")

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import string

np.random.seed(42)
random.seed(42)

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import string

np.random.seed(42)
random.seed(42)

print("=" * 70)
print("ГЕНЕРАЦИЯ ДАННЫХ ДЛЯ ПОРТФОЛИО DATA ANALYST")
print("ШВЕЙНАЯ ПРОМЫШЛЕННОСТЬ (полная версия)")
print("=" * 70)

# ============================================
# 1. Товары (articles) с размерами и артикулами
# ============================================
n_articles = 5000

product_types = ['Dress', 'Trousers', 'Blouse', 'Shirt', 'Jacket', 'Skirt', 'Coat', 'Sweater', 'Jeans']
colors = ['Black', 'White', 'Red', 'Blue', 'Green', 'Pink', 'Grey', 'Navy Blue', 'Beige', 'Brown', 'Purple', 'Yellow']
seasons = ['Spring', 'Summer', 'Autumn', 'Winter']

# РОССИЙСКАЯ РАЗМЕРНАЯ СЕТКА
size_mapping = {
    'Dress': ['40', '42', '44', '46', '48', '50'],
    'Trousers': ['40', '42', '44', '46', '48', '50', '52', '54'],
    'Blouse': ['40', '42', '44', '46', '48', '50'],
    'Shirt': ['40', '42', '44', '46', '48', '50', '52'],
    'Jacket': ['40', '42', '44', '46', '48', '50', '52', '54'],
    'Skirt': ['40', '42', '44', '46', '48', '50'],
    'Coat': ['42', '44', '46', '48', '50', '52', '54', '56'],
    'Sweater': ['40', '42', '44', '46', '48', '50', '52'],
    'Jeans': ['40', '42', '44', '46', '48', '50', '52', '54']
}

def generate_sku():
    letters = ''.join(random.choices(string.ascii_uppercase, k=3))
    numbers = ''.join(random.choices(string.digits, k=5))
    return f"{letters}{numbers}"

articles = []
for i in range(1, n_articles + 1):
    product_type = np.random.choice(product_types)
    available_sizes = size_mapping[product_type]

    # ИСПРАВЛЕНО: используем round() как функцию
    price = round(np.random.uniform(500, 15000), 0)

    articles.append({
        'article_id': i,
        'sku': generate_sku(),
        'product_type_name': product_type,
        'colour_group_name': np.random.choice(colors),
        'season': np.random.choice(seasons),
        'size': np.random.choice(available_sizes),
        'price': float(price)
    })

articles = pd.DataFrame(articles)
articles['cost'] = (articles['price'] * np.random.uniform(0.4, 0.6, n_articles)).round(0)
articles['margin'] = articles['price'] - articles['cost']
articles['margin_percent'] = (articles['margin'] / articles['price'] * 100).round(1)

print(f"✅ Товаров: {len(articles):,}")
print(f"   Размеры: {sorted(articles['size'].unique())}")

# ============================================
# 2. Клиенты (ТОЛЬКО для собственного сайта и офлайн)
# ============================================
n_customers_known = 15000
customer_sizes = ['40', '42', '44', '46', '48', '50', '52', '54', '56']

customers = []
for i in range(1, n_customers_known + 1):
    reg_day = np.random.randint(0, 100)
    reg_date = datetime(2025, 1, 1) + timedelta(days=reg_day)

    customers.append({
        'customer_id': i,
        'age': np.random.randint(18, 70),
        'gender': np.random.choice(['F', 'M', 'Unisex'], p=[0.6, 0.3, 0.1]),
        'typical_size': np.random.choice(customer_sizes),
        'city': np.random.choice(['Москва', 'СПб', 'Новосибирск', 'Екатеринбург', 'Казань', 'Нижний Новгород', 'Челябинск']),
        'club_member_status': np.random.choice(['ACTIVE', 'INACTIVE', 'PREMIUM'], p=[0.5, 0.3, 0.2]),
        'registration_date': reg_date
    })

customers = pd.DataFrame(customers)
print(f"✅ Клиентов с известными данными: {len(customers):,}")

# ============================================
# 3. Склады
# ============================================
warehouses = [
    'Москва (Центральный)',
    'СПб (Северо-Запад)',
    'Новосибирск (Сибирь)',
    'Казань (Поволжье)',
    'Екатеринбург (Урал)',
    'Ростов-на-Дону (Южный)'
]

# ============================================
# 4. Транзакции (100,000 продаж) за 2025 год
# ============================================
n_transactions = 100000
start_date = datetime(2025, 1, 1)

channels = ['Ozon', 'Wildberries', 'Website', 'Offline']

transactions = []
for _ in range(n_transactions):
    channel = np.random.choice(channels, p=[0.35, 0.35, 0.2, 0.1])

    if channel in ['Website', 'Offline']:
        customer_id = np.random.randint(1, n_customers_known + 1)
    else:
        customer_id = 0

    purchase_day = np.random.randint(0, 365)
    purchase_date = start_date + timedelta(days=purchase_day)

    transactions.append({
        't_dat': purchase_date,
        'customer_id': customer_id,
        'article_id': np.random.randint(1, n_articles + 1),
        'sales_channel': channel
    })

transactions = pd.DataFrame(transactions)

# Добавляем информацию о товаре
transactions = transactions.merge(
    articles[['article_id', 'product_type_name', 'colour_group_name', 'season', 'size', 'price', 'cost', 'margin', 'sku']],
    on='article_id',
    how='left'
)

# Для собственных каналов — добавляем данные о клиенте
transactions_with_customer = transactions[transactions['customer_id'] > 0].merge(
    customers[['customer_id', 'typical_size', 'age', 'gender', 'city']],
    on='customer_id',
    how='left'
)

transactions_anonymous = transactions[transactions['customer_id'] == 0].copy()
transactions_anonymous['typical_size'] = np.random.choice(customer_sizes, len(transactions_anonymous))
transactions_anonymous['age'] = np.nan
transactions_anonymous['gender'] = np.nan
transactions_anonymous['city'] = np.random.choice(['Москва', 'СПб', 'Новосибирск', 'Екатеринбург', 'Казань', 'Ростов-на-Дону'], len(transactions_anonymous))

transactions = pd.concat([transactions_with_customer, transactions_anonymous], ignore_index=True)

# Добавляем склад отгрузки
transactions['warehouse'] = transactions['city'].apply(
    lambda x: np.random.choice(warehouses, p=[0.35, 0.2, 0.15, 0.12, 0.1, 0.08]) if pd.notna(x) else np.random.choice(warehouses)
)

print(f"✅ Транзакций: {len(transactions):,}")
print(f"   - Маркетплейсы (анонимно): {len(transactions[transactions['customer_id'] == 0]):,}")
print(f"   - Собственные каналы: {len(transactions[transactions['customer_id'] > 0]):,}")

# ============================================
# 5. Симуляция возвратов
# ============================================
def get_return_prob(product_type, sales_channel, size, typical_size, colour):
    prob = 0.08

    if product_type == 'Dress':
        prob = 0.35
    elif product_type in ['Blouse', 'Jacket']:
        prob = 0.20
    elif product_type in ['Coat', 'Sweater']:
        prob = 0.12
    elif product_type == 'Jeans':
        prob = 0.15

    if size != typical_size:
        prob += 0.25

    if colour in ['Pink', 'Yellow', 'Red']:
        prob += 0.05

    if sales_channel in ['Ozon', 'Wildberries']:
        prob *= 1.4
    elif sales_channel == 'Website':
        prob *= 1.2

    return min(prob, 0.80)

transactions['return_prob'] = transactions.apply(
    lambda x: get_return_prob(x['product_type_name'], x['sales_channel'],
                              x['size'], x['typical_size'], x['colour_group_name']), axis=1
)
transactions['is_returned'] = np.random.random(len(transactions)) < transactions['return_prob']

# Дата возврата (1-14 дней после покупки)
transactions['return_date'] = pd.NaT
return_mask = transactions['is_returned']
transactions.loc[return_mask, 'return_date'] = transactions.loc[return_mask, 't_dat'].apply(
    lambda x: x + timedelta(days=np.random.randint(1, 15))
)

# Убыток от возврата
transactions['return_loss'] = transactions.apply(
    lambda x: x['margin'] if x['is_returned'] else 0, axis=1
)

print(f"   - Возвратов: {transactions['is_returned'].sum():,} ({transactions['is_returned'].mean()*100:.1f}%)")
print(f"   - Убыток от возвратов: {transactions['return_loss'].sum():,.0f} руб.")

# ============================================
# 6. Анализ возвратов по каналам
# ============================================
print("\n📊 ВОЗВРАТЫ ПО КАНАЛАМ ПРОДАЖ:")
channel_stats = transactions.groupby('sales_channel').agg({
    'is_returned': ['count', 'mean']
}).round(3)
channel_stats.columns = ['orders', 'return_rate']
channel_stats['return_rate'] = (channel_stats['return_rate'] * 100).round(1)
channel_stats = channel_stats.sort_values('return_rate', ascending=False)
print(channel_stats)

# ============================================
# 7. Анализ возвратов по размерам
# ============================================
print("\n📊 ВОЗВРАТЫ ПО РАЗМЕРАМ:")
size_stats = transactions.groupby('size').agg({
    'is_returned': ['count', 'mean']
}).round(3)
size_stats.columns = ['orders', 'return_rate']
size_stats['return_rate'] = (size_stats['return_rate'] * 100).round(1)
size_stats = size_stats.sort_values('return_rate', ascending=False)
print(size_stats.head(10))

# ============================================
# 8. Анализ возвратов по цветам
# ============================================
print("\n📊 ВОЗВРАТЫ ПО ЦВЕТАМ:")
color_stats = transactions.groupby('colour_group_name').agg({
    'is_returned': ['count', 'mean']
}).round(3)
color_stats.columns = ['orders', 'return_rate']
color_stats['return_rate'] = (color_stats['return_rate'] * 100).round(1)
color_stats = color_stats.sort_values('return_rate', ascending=False)
print(color_stats.head(10))

# ============================================
# 9. Остатки на складах (ОПТИМИЗИРОВАННАЯ ВЕРСИЯ)
# ============================================

print("\n🔄 Создание остатков на складах...")

# Агрегируем продажи по товарам и складам
sales_count = transactions.groupby(['article_id', 'warehouse']).size().reset_index(name='sold_count')

# Создаём все комбинации товаров и складов сразу (вместо цикла)
all_combinations = []
for warehouse in warehouses:
    temp = articles[['article_id', 'sku', 'product_type_name', 'colour_group_name', 'size', 'price', 'cost', 'margin']].copy()
    temp['warehouse'] = warehouse
    all_combinations.append(temp)

stock = pd.concat(all_combinations, ignore_index=True)

# Добавляем продажи
stock = stock.merge(sales_count, on=['article_id', 'warehouse'], how='left')
stock['sold_count'] = stock['sold_count'].fillna(0).astype(int)

# Векторизованная генерация остатков (без цикла по строкам)
def generate_stock_vectorized(row):
    sold = row['sold_count']
    if sold == 0:
        return np.random.randint(10, 80)
    elif sold < 5:
        return np.random.randint(5, 40)
    else:
        return int(sold * np.random.uniform(0.1, 0.4))

# Применяем функцию к каждой строке (быстрее, чем цикл)
stock['current_stock'] = stock.apply(generate_stock_vectorized, axis=1)
stock['stock_value'] = stock['current_stock'] * stock['cost']

print(f"✅ Остатков на складах: {stock['current_stock'].sum():,} шт.")
print(f"✅ Стоимость остатков: {stock['stock_value'].sum():,.0f} руб.")

# Распределение по складам
warehouse_stock = stock.groupby('warehouse')['current_stock'].sum().sort_values(ascending=False)
print("\n📦 ОСТАТКИ ПО СКЛАДАМ:")
for wh, qty in warehouse_stock.items():
    print(f"   {wh}: {qty:,} шт.")



# ============================================
# 10. Сохраняем файлы в CSV (с правильной локалью)
# ============================================

# Сохраняем с русской локалью и правильным разделителем
articles.to_csv('articles.csv', index=False, encoding='utf-8-sig', sep=';', decimal=',')
customers.to_csv('customers.csv', index=False, encoding='utf-8-sig', sep=';', decimal=',')
transactions.to_csv('transactions.csv', index=False, encoding='utf-8-sig', sep=';', decimal=',')
stock.to_csv('stock.csv', index=False, encoding='utf-8-sig', sep=';', decimal=',')

print("\n" + "=" * 70)
print("ФАЙЛЫ СОХРАНЕНЫ В ФОРМАТЕ CSV!")
print("=" * 70)
print("📁 articles.csv      - товары (5 000 SKU, размеры 40-56)")
print("📁 customers.csv     - клиенты (15 000, с размерами и городом)")
print("📁 transactions.csv  - продажи (100 000 записей)")
print("📁 stock.csv         - остатки по 6 складам")
print("=" * 70)

# Проверка: показываем путь к файлам
import os
print(f"\n📂 Файлы сохранены в папке: {os.getcwd()}")

# ============================================
# 11. Выводы для портфолио
# ============================================
print("\n" + "=" * 70)
print("ВЫВОДЫ ДЛЯ ПОРТФОЛИО")
print("=" * 70)

print(f"""
📌 КЛЮЧЕВЫЕ ОСОБЕННОСТИ ДАННЫХ:

1. РАЗМЕРНАЯ СЕТКА:
   • Российские размеры: 40-56
   • Разная сетка для разных типов товаров

2. СКЛАДЫ:
   • 6 региональных складов
   • Общая стоимость остатков: {stock['stock_value'].sum():,.0f} руб.

3. КАНАЛЫ ПРОДАЖ:
   • Маркетплейсы (Ozon, WB): customer_id = 0
   • Собственные каналы: полные данные о клиентах

4. ВОЗВРАТЫ:
   • Общая доля: {transactions['is_returned'].mean()*100:.1f}%
   • Самый высокий канал: {channel_stats.index[0]} ({channel_stats.iloc[0]['return_rate']:.1f}%)
   • Самый проблемный размер: {size_stats.index[0]} ({size_stats.iloc[0]['return_rate']:.1f}%)
   • Самый проблемный цвет: {color_stats.index[0]} ({color_stats.iloc[0]['return_rate']:.1f}%)
""")

print("\n✅ ГОТОВО! Файлы CSV созданы.")


print("DEBUG: Дошел до сохранения файлов")
print(f"Статей: {len(articles)}")
print(f"Клиентов: {len(customers)}")
print(f"Транзакций: {len(transactions)}")
print(f"Остатков: {len(stock)}")